# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
Dataset Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

This resource documents mass spectrometry-based analysis of protein abundance, modifications, and sequences in extracellular vesicles isolated from stimulated human mast cells. We will interactively explore its structure and records using their `@id` fields following Croissant best practices.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset (this only loads metadata, not all data files yet)
dataset = mlc.Dataset(croissant_url)

# Print high level dataset metadata
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets and fields. Each entity is referenced by its `@id` field, which uniquely identifies it within the Croissant schema.

In [ ]:
# List all available record sets with their @id and fields
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Available Fields (@id: name):")
    for field in rs.fields:
        print(f"    {field.id}: {field.name}")
    print('')

# Pick the main record set for subsequent steps
if record_sets:
    record_set_id = record_sets[0].id
    print(f"Primary RecordSet chosen for extraction: '{record_set_id}'")
else:
    raise Exception("No record sets found in dataset.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.

Remember: all references to record sets or fields should be by their `@id`.

In [ ]:
# Extract records from each record set into a dataframe
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    # Using the record set @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns for record set: {rs_id}")

main_df = dataframes[record_set_id]
print("\nAvailable columns (@id):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore the dataset with some basic EDA by referencing columns by their `@id` as per Croissant schema best practices.

We'll perform:
- Filtering for high-abundance proteins (using a numeric field).
- Normalizing a numeric field.
- Grouping by a categorical attribute if present.

In [ ]:
# For demonstration, let's try to automatically select a numeric field and a grouping field

df = main_df

# Try to pick likely numeric and group fields by name
import numpy as np

# Helper: choose field ids by looking for typical biological numeric and categorical names
def guess_field(fields, keys):
    for field in fields:
        for key in keys:
            if key.lower() in field.lower():
                return field
    return None

numeric_candidates = ['abundance', 'coverage', 'count', 'value', 'MW', 'peptide', 'normalized']
group_candidates = ['description', 'gene', 'protein', 'category', 'cell']

numeric_field_id = guess_field(df.columns, numeric_candidates)
if not numeric_field_id:
    # fallback: first numeric column
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field_id = col
            break

group_field_id = guess_field(df.columns, group_candidates)

if numeric_field_id is None:
    raise ValueError("No apparent numeric column found for EDA!")

print(f"Using numeric field: '{numeric_field_id}'")
if group_field_id:
    print(f"Grouping field: '{group_field_id}'")

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering for high values
threshold = np.nanpercentile(df[numeric_field_id], 75)  # Top 25% abundance or value
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top quartile):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

# Optional: Grouped analytic (mean per category if possible)
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped means of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped.head())

## 5. Visualization
Let's quickly visualize the numeric field distribution and its normalized counterpart for high-abundance records.

If grouping is available, show a bar plot of means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution plot (raw and normalized)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, ax=axes[0], kde=True)
axes[0].set_title(f"Distribution of {numeric_field_id} (top quartile)")

sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, ax=axes[1], kde=True, color="orange")
axes[1].set_title(f"Normalized {numeric_field_id} (Z-score)")
plt.tight_layout()
plt.show()

# Optional: Bar plot by group
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(10, 5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
    plt.title(f"Mean {numeric_field_id} per {group_field_id} (top quartile)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the [FAIR\u2072 dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) with `mlcroissant`, listed record sets and fields by their `@id`, loaded a primary data table into a DataFrame, and conducted basic EDA by filtering, normalizing, and visualizing a main numeric field.

All exploration referenced record sets and fields *by their `@id`*, consistent with Croissant and semantic best practices, ensuring reproducibility and schema traceability.

Next steps might include deeper biological/statistical investigation, direct linking of sample annotations or protein identifiers with external resources, or extending the notebook based on specific research questions.